In [1]:
!git clone https://github.com/clayton-h-costa/pv_fault_dataset.git

Cloning into 'pv_fault_dataset'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 24 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 26.69 MiB | 2.87 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from scipy.io import loadmat
warnings.filterwarnings('ignore')

# Set ggplot style
plt.style.use('ggplot')

# ML and Deep Learning
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_recall_fscore_support,
    cohen_kappa_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    auc,
    make_scorer
)
from sklearn.ensemble import RandomForestClassifier
import joblib
import mlflow
from typing import Tuple, Dict, List, Any, Optional
import numpy.typing as npt

In [3]:
mlflow.set_experiment("Solar_Fault_RF")

2026/03/28 04:45:14 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/28 04:45:14 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/28 04:45:14 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/28 04:45:14 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/28 04:45:14 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/28 04:45:14 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/28 04:45:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/28 04:45:15 INFO mlflow.store.db.utils: Updating database tables
2026/03/28 04:45:15 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/28 04:45:15 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/03/28 04:45:15 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2026/03/28 04:4

<Experiment: artifact_location=('/Users/seyedrumaiz/Library/Mobile '
 'Documents/com~apple~CloudDocs/DSGP/solar-panel-fault-mapping/notebooks/fault-detection/random_forest/mlruns/1'), creation_time=1774653315417, experiment_id='1', last_update_time=1774653315417, lifecycle_stage='active', name='Solar_Fault_RF', tags={}>

In [ ]:
def load_data() -> pd.DataFrame:
    """
    Load and merge electrical and ambient PV fault datasets from
    .mat files.

    Returns:
        pd.DataFrame: Cleaned dataframe containing electrical measurements,
        environmental variables, and mapped fault labels.
    """
    elec_data = loadmat("pv_fault_dataset/dataset_elec.mat")
    amb_data  = loadmat("pv_fault_dataset/dataset_amb.mat")

    df = pd.DataFrame({
        'vdc1': elec_data['vdc1'].flatten(),
        'vdc2': elec_data['vdc2'].flatten(),
        'idc1': elec_data['idc1'].flatten(),
        'idc2': elec_data['idc2'].flatten(),
        'irradiance': amb_data['irr'].flatten(),
        'temperature': amb_data['pvt'].flatten(),
        'fault_label': amb_data['f_nv'].flatten()
    })

    # Remove degradation label
    df = df[df['fault_label'] != 2].copy()

    fault_names = {
        0: 'Normal Operation',
        1: 'Short-Circuit',
        3: 'Open Circuit',
        4: 'Shadowing'
    }

    df['fault_label'] = df['fault_label'].map(fault_names)
    return df

In [ ]:
def feature_engineering(df: pd.DataFrame) -> Tuple[npt.NDArray[np.float64], npt.NDArray[Any], List[str]]:
    """
    Generate additional power and ratio features for model training.

    Args:
        df (pd.DataFrame): Raw dataframe containing PV measurements.

    Returns:
        X (np.ndarray): Feature matrix.
        y (np.ndarray): Target labels.
        feature_names (list[str]): Name of engineered features.
    """
    df['power_string1'] = df['vdc1'] * df['idc1']
    df['power_string2'] = df['vdc2'] * df['idc2']
    df['total_power'] = df['power_string1'] + df['power_string2']
    df['voltage_ratio'] = df['vdc1'] / df['vdc2']
    df['current_ratio'] = df['idc1'] / df['idc2']

    X = df.drop("fault_label", axis=1).values
    y = df["fault_label"].values
    feature_names = df.drop("fault_label", axis=1).columns.tolist()

    return X, y, feature_names

In [ ]:
def split_data(X: npt.NDArray[Any],
               y: npt.NDArray[Any]) -> Tuple[npt.NDArray[Any], npt.NDArray[Any], npt.NDArray[Any], npt.NDArray[Any]]:
    """
    Split dataset into stratified train and test sets.

    Args:
        X (np.ndarray): Feature matrix.
        y (np.ndarray): Target labels.

    Returns:
        X_train, X_test, y_train, y_test: Split datasets.
    """
    return train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

In [ ]:
def compute_weights(y_train: npt.NDArray[Any]) -> Dict[Any, float]:
    """
    Compute class weights to handle imbalanced dataset.

    Args:
        y_train (np.ndarray): Training labels.

    Returns:
        dict: Mapping of class label to weight.
    """
    classes = np.unique(y_train)
    weights = compute_class_weight("balanced", classes=classes, y=y_train)
    return dict(zip(classes, weights))

In [ ]:
def train_random_forest(X_train: npt.NDArray[Any],
                        y_train: npt.NDArray[Any],
                        class_weights,
                        params=None) -> RandomForestClassifier:
    """
    Train Random Forest classifier.

    Args:
        X_train (np.ndarray): Training features.
        y_train (np.ndarray): Training labels.
        class_weights (dict): Class weights.
        params (dict, optional): Extra RF hyperparameters.

    Returns:
        RandomForestClassifier: Trained model.
    """
    if params is None:
        params = {}

    rf = RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight=class_weights,
        **params
    )

    rf.fit(X_train, y_train)
    return rf

In [ ]:
def evaluate_model(model: RandomForestClassifier,
                   X_test: npt.NDArray[Any],
                   y_test: npt.NDArray[Any]) -> Tuple[Dict[str, float], npt.NDArray[Any], npt.NDArray[Any], npt.NDArray[Any], npt.NDArray[Any]]:
    """
    Evaluate classification model using multiple metrics.

    Args:
        model (RandomForestClassifier): Trained model.
        X_test (np.ndarray): Test features.
        y_test (np.ndarray): Test labels.

    Returns:
        metrics (dict): Evaluation metrics.
        y_pred (np.ndarray): Predicted labels.
        y_proba (np.ndarray): Prediction probabilities.
        class_order (np.ndarray): Class ordering used by model.
        y_test_bin (np.ndarray): Binarized labels.
    """
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    class_order = model.classes_

    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    kappa = cohen_kappa_score(y_test, y_pred)

    y_test_bin = label_binarize(y_test, classes=class_order)

    roc_auc = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
    pr_auc  = average_precision_score(y_test_bin, y_proba, average="macro")

    metrics = {
        "accuracy": acc,
        "precision_macro": prec,
        "recall_macro": rec,
        "f1_macro": f1,
        "cohens_kappa": kappa,
        "roc_auc_macro": roc_auc,
        "pr_auc_macro": pr_auc
    }

    return metrics, y_pred, y_proba, class_order, y_test_bin

In [ ]:
def log_classification_report(y_test, y_pred, classes) -> None:
    """
    Save classification report and log to MLflow.

    Args:
        y_test (np.ndarray): True labels.
        y_pred (np.ndarray): Predicted labels.
        classes (np.ndarray): Class names.
    
    Returns:
        None
    """
    report_dict = classification_report(
        y_test,
        y_pred,
        target_names=classes,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv("classification_report.csv")

    mlflow.log_artifact("classification_report.csv")

In [ ]:
def plot_confusion(y_test: npt.NDArray[Any],
                   y_pred: npt.NDArray[Any],
                   classes: npt.NDArray[Any]) -> None:
    """
    Plot confusion matrix and log artifact.

    Args:
        y_test (np.ndarray): True labels.
        y_pred (np.ndarray): Predictions.
        classes (np.ndarray): Class labels

    Returns:
        None
    """
    cm = confusion_matrix(y_test,y_pred,labels=classes)
    disp = ConfusionMatrixDisplay(cm)
    disp.plot(cmap="Reds")
    plt.title("Confusion Matrix")
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifact("confusion_matrix.png")
    plt.close()

In [ ]:
def plot_roc(
    y_bin: npt.NDArray[Any],
    y_proba: npt.NDArray[Any],
    classes: npt.NDArray[Any]
) -> None:
    """
    Plot ROC curves for all classes.

    Args:
        y_bin (np.ndarray): Binarized labels.
        y_proba (np.ndarray): Prediction probabilities.
        classes (np.ndarray): Class labels.

    Returns:
        None
    """
    plt.figure()

    for i, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
        roc_auc_val = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{cls} (AUC = {roc_auc_val:.2f})")

    plt.plot([0,1], [0,1], linestyle="--")
    plt.legend()
    plt.title("ROC Curves")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")

    plt.savefig("roc.png")
    mlflow.log_artifact("roc.png")
    plt.close()

In [ ]:
def plot_pr(
    y_bin: npt.NDArray[Any],
    y_proba: npt.NDArray[Any],
    classes: npt.NDArray[Any]
) -> None:
    """
    Plot precision-recall curves for all classes.

    Args:
        y_bin (np.ndarray): Binarized labels.
        y_proba (np.ndarray): Prediction probabilities.
        classes (np.ndarray): Class labels.
    """
    plt.figure()

    for i, cls in enumerate(classes):
        precision, recall, _ = precision_recall_curve(y_bin[:, i], y_proba[:, i])
        pr_auc_val = auc(recall, precision)   # ← USING auc AGAIN
        plt.plot(recall, precision, label=f"{cls} (AUC = {pr_auc_val:.2f})")

    plt.legend()
    plt.title("Precision-Recall Curves")
    plt.xlabel("Recall")
    plt.ylabel("Precision")

    plt.savefig("pr.png")
    mlflow.log_artifact("pr.png")
    plt.close()

In [ ]:
def log_auc_per_class(
    y_bin: npt.NDArray[Any],
    y_proba: npt.NDArray[Any],
    classes: npt.NDArray[Any]
) -> None:
    """
    Log ROC-AUC and PR-AUC per class to MLflow.

    Args:
        y_bin (np.ndarray): Binarized labels.
        y_proba (np.ndarray): Prediction probabilities.
        classes (np.ndarray): Class labels.
    """
    roc_auc_per_class = {}
    pr_auc_per_class  = {}

    for i, cls in enumerate(classes):
        roc_auc_per_class[f"roc_auc_{cls}"] = roc_auc_score(y_bin[:, i], y_proba[:, i])
        pr_auc_per_class[f"pr_auc_{cls}"]   = average_precision_score(y_bin[:, i], y_proba[:, i])

    mlflow.log_metrics(roc_auc_per_class)
    mlflow.log_metrics(pr_auc_per_class)

In [ ]:
def plot_feature_importance(
    model: RandomForestClassifier,
    feature_names: List[str]
) -> None:
    """
    Plot and log top feature importances.

    Args:
        model (RandomForestClassifier): Trained model.
        feature_names (list[str]): Feature names.
    """    
    fi = pd.Series(model.feature_importances_,index=feature_names)\
            .sort_values(ascending=False)

    fi.head(10).plot(kind="bar")
    plt.title("Feature Importance")
    plt.savefig("feature_importance.png")
    mlflow.log_artifact("feature_importance.png")
    plt.close()

In [ ]:
def cross_validate_model(
    model: RandomForestClassifier,
    X_train: npt.NDArray[Any],
    y_train: npt.NDArray[Any]
) -> Dict[str, Any]:
    """
    Perform stratified K-Fold cross validation.

    Args:
        model (RandomForestClassifier): Model to validate.
        X_train (np.ndarray): Training features.
        y_train (np.ndarray): Training labels.

    Returns:
        dict: Cross-validation results.
    """
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    kappa_scorer = make_scorer(cohen_kappa_score)

    scoring = {
        "accuracy": "accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "f1_macro": "f1_macro",
        "roc_auc_ovr": "roc_auc_ovr",
        "kappa": kappa_scorer
    }

    results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    # log CV metrics mean
    for key in results:
        if "test_" in key:
            mlflow.log_metric(f"cv_{key}", results[key].mean())

    return results

In [ ]:
def tune_hyperparameters(
    X_train: npt.NDArray[Any],
    y_train: npt.NDArray[Any],
    class_weights: Dict[Any, float]
) -> Tuple[RandomForestClassifier, Dict[str, Any]]:
    """
    Tune RandomForest hyperparameters using RandomizedSearchCV.

    Args:
        X_train (np.ndarray): Training features.
        y_train (np.ndarray): Training labels.
        class_weights (dict): Class weights.

    Returns:
        best_model (RandomForestClassifier): Best tuned model.
        best_params (dict): Best hyperparameters.
    """    
    param_dist = {
        "n_estimators": [100,150,200,300,400],
        "max_depth": [None,10,20,30,40],
        "min_samples_split": [2,3,4,5],
        "min_samples_leaf": [1,2,3],
        "max_features": ["sqrt","log2", None]
    }

    base_rf = RandomForestClassifier(
        class_weight=class_weights,
        random_state=42,
        n_jobs=-1
    )

    search = RandomizedSearchCV(
        base_rf,
        param_dist,
        n_iter=15,
        cv=3,
        scoring="roc_auc_ovr",
        n_jobs=-1,
        random_state=42
    )

    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_params_

In [ ]:
def compare_models(
    base_metrics: Dict[str, float],
    tuned_metrics: Dict[str, float]
) -> None:
    """
    Compare base vs tuned model metrics and log results.

    Args:
        base_metrics (dict): Metrics of base model.
        tuned_metrics (dict): Metrics of tuned model.

    Returns:
        None
    """
    df = pd.DataFrame([base_metrics,tuned_metrics],index=["Base","Tuned"])
    print(df)
    df.to_csv("comparison.csv")
    mlflow.log_artifact("comparison.csv")

In [ ]:
def save_model_locally(
    model: RandomForestClassifier,
    filename: str = "best_model.pkl"
) -> None:
    """
    Save trained model locally and log artifact.

    Args:
        model (RandomForestClassifier): Model to save.
        filename (str): Output filename.
    """
    joblib.dump(model, filename)
    mlflow.log_artifact(filename)

In [ ]:
def run_experiment(
    run_name: str,
    model: RandomForestClassifier,
    X_train: npt.NDArray[Any],
    X_test: npt.NDArray[Any],
    y_train: npt.NDArray[Any],
    y_test: npt.NDArray[Any],
    feature_names: List[str],
    params: Optional[Dict[str, Any]] = None
) -> Dict[str, float]:
    """
    Run full MLflow experiment pipeline.

    Args:
        run_name (str): MLflow run name.
        model (RandomForestClassifier): Model to evaluate.
        X_train (np.ndarray): Training features.
        X_test (np.ndarray): Test features.
        y_train (np.ndarray): Training labels.
        y_test (np.ndarray): Test labels.
        feature_names (list[str]): Feature names.
        params (dict, optional): Hyperparameters to log.

    Returns:
        dict: Evaluation metrics.
    """

    with mlflow.start_run(run_name=run_name):

        if params is not None:
            mlflow.log_params(params)

        # Cross-validation first
        cross_validate_model(model, X_train, y_train)

        # Evaluation
        metrics, y_pred, y_proba, classes, y_bin = evaluate_model(model, X_test, y_test)

        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, "model")

        # Extra logging
        log_classification_report(y_test, y_pred, classes)
        log_auc_per_class(y_bin, y_proba, classes)

        # Plots
        plot_confusion(y_test, y_pred, classes)
        plot_roc(y_bin, y_proba, classes)
        plot_pr(y_bin, y_proba, classes)
        plot_feature_importance(model, feature_names)

        return metrics

In [ ]:
def main() -> None:
    """
    Entry point for full ML training pipeline.
    """

    # Load + feature engineering
    print("Loading data.")
    df = load_data()
    X, y, feature_names = feature_engineering(df)

    X_train, X_test, y_train, y_test = split_data(X, y)
    class_weights = compute_weights(y_train)
    print("Split data into train-test splits.")

    # Base model run
    base_model = train_random_forest(X_train, y_train, class_weights)
    print("Trained base model.")

    base_metrics = run_experiment(
        "Base_RF",
        base_model,
        X_train, X_test, y_train, y_test,
        feature_names
    )
    print("Evaluated base model.")

    # Tuned model run
    best_model, best_params = tune_hyperparameters(X_train, y_train, class_weights)
    print("Tuned base model.")

    tuned_metrics = run_experiment(
        "Tuned_RF",
        best_model,
        X_train, X_test, y_train, y_test,
        feature_names,
        params=best_params
    )
    print("Evaluated best model.")
    # Save best tuned model locally
    save_model_locally(best_model)
    print("Comparing both models.")
    compare_models(base_metrics, tuned_metrics)

In [22]:
if __name__ == "__main__":
    main()

Loading data.
Split data into train-test splits.
Trained base model.


2026/03/28 04:48:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Evaluated base model.
Tuned base model.


2026/03/28 07:33:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Evaluated best model.
Comparing both models.
       accuracy  precision_macro  recall_macro  f1_macro  cohens_kappa  \
Base   0.999168         0.998795      0.997796  0.998295      0.996713   
Tuned  0.999149         0.998152      0.998819  0.998485      0.996643   

       roc_auc_macro  pr_auc_macro  
Base        0.999996      0.999977  
Tuned       0.999997      0.999983  
